# ClosetScan: video → wardrobe
Run cells in order in Google Colab or local Jupyter with Python 3.10+ and FFmpeg/ffprobe.
In Colab, upload the source ZIP when prompted (the "Source code (zip)" asset on the release page).
Locally, launch Jupyter from a clone or set `PROJECT_DIR` below.
With `USE_DEMO=True` the demo walkthrough downloads automatically; it is published separately from the repository.
For your own footage, record it narrated — see [Recording a walkthrough](https://github.com/wgaostudio/closetscan#recording-a-walkthrough).

The default histogram backend exercises the pipeline without model downloads; it is a smoke test, not a reliable measure of garment recognition. Choose `dinov2` for real footage.

Optional AI stages send images and transcript text to OpenRouter and its providers and incur charges. Generated plates are reconstructions, not photographs. The default run makes no paid API calls.

In [ ]:
from pathlib import Path
import sys, os, shutil, tempfile, subprocess, zipfile

USE_DEMO = True
RUN_AI = False
INCLUDE_NARRATION = False
EMBEDDER = 'hist'  # 'dinov2' downloads model weights; a GPU helps
VISION_MODEL = ''
IMAGE_MODEL = ''
PROJECT_DIR = ''  # local only: blank uses the notebook working directory
VIDEO_PATH = ''   # local only: required when USE_DEMO=False
WORKSPACE_DIR = ''  # blank creates a temporary directory; choose a new folder to keep outputs

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if EMBEDDER not in ('hist', 'dinov2'):
    raise ValueError('EMBEDDER must be hist or dinov2')
if INCLUDE_NARRATION and not RUN_AI:
    raise ValueError('INCLUDE_NARRATION requires RUN_AI=True')
if RUN_AI and not (VISION_MODEL and IMAGE_MODEL):
    raise ValueError('Set both model IDs before running paid stages')
workspace = Path(WORKSPACE_DIR).expanduser().resolve() if WORKSPACE_DIR else Path(tempfile.mkdtemp(prefix='closetscan-session-'))
workspace.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    uploaded = files.upload()
    archives = [n for n in uploaded if n.endswith('.zip')]
    assert len(archives) == 1, 'Upload exactly one release ZIP'
    root = workspace / 'release'
    root.mkdir()
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (root / member.filename).resolve().is_relative_to(root.resolve()):
                raise ValueError('Unsafe ZIP path')
        archive.extractall(root)
    projects = list(root.rglob('pyproject.toml'))
    assert len(projects) == 1, 'Expected one release project'
    project = projects[0].parent
else:
    project = Path(PROJECT_DIR or Path.cwd()).expanduser().resolve()
assert (project / 'pyproject.toml').is_file(), 'Set PROJECT_DIR to the release folder'
for command in ('ffmpeg', 'ffprobe'):
    assert shutil.which(command), f'{command} is required on PATH'
extras = ['pipeline']
if EMBEDDER == 'dinov2':
    extras.append('vision')
if INCLUDE_NARRATION:
    extras.append('narration')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                str(project) + '[' + ','.join(extras) + ']'], check=True)
def run(module, *args):
    subprocess.run([sys.executable, '-u', '-m', 'closetscan.' + module,
                    *map(str, args)], check=True, cwd=workspace)
print('Installed ClosetScan using', sys.version.split()[0])
print('Environment:', 'Colab' if IN_COLAB else 'local Jupyter')
print('Project:', project)

## Input
Use the demo or choose one walkthrough video. Every execution creates a fresh output directory.

**Talk through the closet while you film your own video.** Speech is the pipeline's strongest segmentation signal: a closet background is constant, so the pauses between utterances mark the handoff from one garment to the next far more reliably than visual change does. A silent recording still processes, but neighbouring garments merge into one, and `INCLUDE_NARRATION` has nothing to transcribe. Say a few words about each piece, pause between items, hold each garment still, and show both sides. Keep the audio track if you trim the clip first. Full guidance: [Recording a walkthrough](https://github.com/wgaostudio/closetscan#recording-a-walkthrough).

In [ ]:
DEMO_VIDEO_URL = 'https://github.com/wgaostudio/closetscan/releases/latest/download/phone-walkthrough.mp4'

if USE_DEMO:
    clip = project / 'demo/phone-walkthrough.mp4'
    if not clip.is_file():
        # The demo is published separately to keep the repository small.
        clip = workspace / 'phone-walkthrough.mp4'
        if not clip.is_file():
            import urllib.request
            print('Fetching the demo walkthrough:', DEMO_VIDEO_URL)
            urllib.request.urlretrieve(DEMO_VIDEO_URL, clip)
elif IN_COLAB:
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload one walkthrough video'
    clip = Path(next(iter(uploaded))).resolve()
else:
    assert VIDEO_PATH, 'Set VIDEO_PATH when USE_DEMO=False'
    clip = Path(VIDEO_PATH).expanduser().resolve()
assert clip.is_file(), f'Video not found: {clip}'
out = workspace / 'catalogue'
out.mkdir()
print('Input:', clip)
print('Output:', out)

In [ ]:
if RUN_AI:
    from closetscan.product_shots import read_key, KEY_FILE
    if not os.environ.get('OPENROUTER_API_KEY') and not Path(KEY_FILE).is_file():
        import getpass
        os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()
    assert read_key(), 'API key is empty'
run('run', clip, '--out', out, '--embedder', EMBEDDER, '--html')

In [ ]:
from IPython.display import Image, display
for sheet in sorted(out.glob('contact_sheet*.jpg')):
    display(Image(filename=str(sheet), width=900))

In [ ]:
if RUN_AI:
    run('dedup', out, '--include-views', '--model', VISION_MODEL)

In [ ]:
if RUN_AI:
    run('product_shots', out, '--model', IMAGE_MODEL)

In [ ]:
if RUN_AI:
    run('attributes', out, '--model', VISION_MODEL)

In [ ]:
if INCLUDE_NARRATION:
    run('narration', out, '--clip', clip, '--model', VISION_MODEL)
run('html_export', out)

## Validate and export
Check the manifest, image files, exported HTML, and MCP query before creating the ZIP.
These checks verify the workflow completes; inspect the contact sheets to judge garment recognition.
Colab downloads the ZIP. Local Jupyter prints its path. Extract it and open `catalogue.html`.

In [ ]:
import json
from closetscan.catalogue import load_catalogue
manifest = json.loads((out / 'manifest.json').read_text())
assert manifest['n_frames'] > 0, 'No frames extracted'
garments = load_catalogue(str(out))
assert garments, 'No garment candidates detected'
if RUN_AI:
    grouping = json.loads((out / 'dedup.json').read_text())
    from closetscan.dedup import check_coverage
    valid_images = {f'row_{i}' for i in range(len(manifest['garments']))}
    valid_images.update(f'row_{i}_alt{j}' for i, row in enumerate(manifest['garments']) for j, _ in enumerate(row.get('views', [])))
    problems = check_coverage(grouping, len(manifest['garments']), valid_images)
    assert not problems, problems
    products = json.loads((out / 'products/products.json').read_text())['images']
    attrs = json.loads((out / 'attributes.json').read_text())['garments']
    for garment in garments:
        for side in ('front', 'back'):
            assert any(p['group'] == garment.index and p['view'] == side and p.get('file') for p in products), f'Missing {side}: {garment.id}'
        assert attrs.get(str(garment.index), {}).get('category'), f'Missing attributes: {garment.id}'
if INCLUDE_NARRATION:
    narration = json.loads((out / 'narration.json').read_text())
    assert narration.get('words'), 'No transcript words'
    assert narration.get('garments'), 'No narration attached to garments'
for garment in garments:
    assert garment.views, f'No images for {garment.id}'
    for view in garment.views:
        path = out / view.file
        assert path.is_file() and path.stat().st_size > 0, f'Missing image: {path}'
assert (out / 'catalogue.html').stat().st_size > 0
request = {'jsonrpc': '2.0', 'id': 1, 'method': 'tools/call',
           'params': {'name': 'list_garments', 'arguments': {'limit': 1}}}
reply = subprocess.run([sys.executable, '-m', 'closetscan.mcp_server', str(out)],
                       input=json.dumps(request) + '\n', text=True,
                       capture_output=True, check=True, cwd=workspace)
result = json.loads(reply.stdout)['result']
assert not result['isError'], result
assert json.loads(result['content'][0]['text'])['total'] == len(garments)
archive = shutil.make_archive(str(out), 'zip', out)
with zipfile.ZipFile(archive) as exported:
    assert exported.testzip() is None
kind = "garments" if RUN_AI else "candidates"
print(f"Validated {manifest['n_frames']} frames and {len(garments)} {kind}; MCP and ZIP checks passed.")
print('Archive:', archive)
if IN_COLAB:
    files.download(archive)